# Miami

In [ ]:
import os
import sys
from pathlib import Path

for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"

ROOT = Path.cwd().resolve()
while not (ROOT / "utils.py").is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Open this notebook inside the code folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from realdata.run_realdata import run_realdata
from utils import ratio_table
from IPython.display import HTML


## Preprocessing

In [ ]:
import numpy as np
import pandas as pd

DATA_DIR = ROOT / "realdata" / "data" / "miami"  # Folder for this dataset's source and processed files.

if not (DATA_DIR / "miami_openml_43093_processed.csv").is_file():
    # all attributes are numeric, so the @data part is plain comma-separated text
    lines = (DATA_DIR / "miami_openml_43093.arff").read_text(encoding="utf-8").splitlines()
    columns = [line.split()[1].strip("'") for line in lines if line.lower().startswith("@attribute")]
    start = next(i for i, line in enumerate(lines) if line.lower().startswith("@data")) + 1
    raw = pd.DataFrame([line.split(",") for line in lines[start:] if line.strip()], columns=columns)
    raw.to_csv(DATA_DIR / "miami_openml_43093.csv", index=False)

    raw = pd.read_csv(DATA_DIR / "miami_openml_43093.csv")

    FEATURES = ["LATITUDE", "LONGITUDE", "LND_SQFOOT", "TOT_LVG_AREA", "SPEC_FEAT_VAL", "RAIL_DIST",  # Input columns used to predict the response.
                "OCEAN_DIST", "WATER_DIST", "CNTR_DIST", "SUBCNTR_DI", "HWY_DIST", "age"]

    data = raw[FEATURES].copy()
    data["LOG_SALE_PRC"] = np.log(raw["SALE_PRC"])  # Response to predict, using the natural log of the sale price.
    data = data.dropna()
    data.to_csv(DATA_DIR / "miami_openml_43093_processed.csv", index=False)

In [ ]:
labeled_sizes = range(100, 1001, 100)  # Labeled sample sizes: 100, 200, ..., 1000.
N = 5000  # Number of unlabeled observations (X only).
reps = 100  # Number of repeated experiments for each setting.
seed = 202605310  # Random seed for the first repetition.
jobs = 12  # Maximum number of repetitions run at the same time.
estimators = [  # Methods to fit to each repeated sample.
    "ata_sbf",
    "ata_csbf",
    "lp_nw", "lp_ll",  # n^(-k/(21*d)), k = 1, ..., 20, capped at 0.8. Repeated values are used once.
    "krr_m",
    "krr_r",
    "ease_nw",
    "ease_ll",
    "ease_m",
    "ease_r",
    "pi",
]

# EASE splits the labeled sample into K groups.
# Fit on K - 1 groups and predict the remaining group.
K = 5
# EASE uses SS-SIR to combine the input columns into r new variables.
# Fit the regression using these r variables.
r = 2
# Split the labeled sample into cv_folds groups to compare tuning candidates.
# For EASE, fit the full estimator on all but one group for each candidate.
# Use the final EASE coefficients to measure prediction error on the remaining group.
cv_folds = 10

summary_miami = {}
for n in labeled_sizes:
    # EASE's SS-SIR splits the response range into H equal-width intervals (slices).
    H = n // 5
    summary_miami[n] = run_realdata(
        "miami",
        estimators=estimators,
        n=n,
        N=N,
        reps=reps,
        seed=seed,
        K=K,
        r=r,
        H=H,
        cv_folds=cv_folds,
        jobs=jobs,
        run_name=f"miami_n{n}_N{N}",
    )[0]

In [ ]:
table_miami = ratio_table(pd.concat(summary_miami.values(), ignore_index=True), "miami")
shown_miami = table_miami.map(lambda value: f"{value:.3f}")
shown_miami = shown_miami.mask(
    table_miami.eq(table_miami.max()), "<b>" + shown_miami + "</b>"
)
HTML(shown_miami.to_html(escape=False))